# 5. Advanced RAG Techniques (Các kỹ thuật nâng cao)

## 📖 Lý thuyết
Trong dự án RAG thực tế, hệ thống LangChain cơ bản không giải quyết được các lỗi phát sinh. 
Qua file `qa_service.py` của dự án SGU, chúng ta rút ra các kỹ thuật hay:
1. **Dynamic Top-K Retrieval**: Tự tăng dần số lượng kết quả (k) khi câu hỏi có dạng "tổng quát" (ví dụ hỏi mục tiêu, chuẩn đầu ra).
2. **Graceful Degradation (Chế độ Fallback)**: Nếu LLM bị sập hoặc quá tải (Rate limit), thay vì báo lỗi đỏ chót, hệ thống tự động trả trực tiếp đoạn văn chứa thông tin để người dùng đọc tạm (`_build_retrieval_only_answer`).
3. **Citation & Provenance**: Phân tách Metadata thủ công cực tỉ mỉ từ Langhain `Document` để hiển thị Nguồn trích dẫn (tên file, số trang).


## 💻 Code mẫu: Dynamic Top-K và Metadata Trích dẫn

In [1]:
import unicodedata
import re

# 1. Kỹ thuật DYNAMIC TOP-K (Mô phỏng _should_expand_retrieval)
def should_expand_retrieval(question: str) -> bool:
    lowered = question.casefold()
    without_marks = "".join(c for c in unicodedata.normalize("NFD", lowered) if unicodedata.category(c) != "Mn")
    normalized = re.sub(r"[^a-z0-9]+", " ", without_marks).strip()
    
    broad_markers = ("muc tieu", "chuan dau ra", "noi dung", "bao gom")
    return any(marker in normalized for marker in broad_markers)

questions = [
    "Điều 5 quy chế học vụ nói gì?", 
    "Hãy liệt kê các mục tiêu của môn Cấu trúc dữ liệu"
]

base_k = 4
for q in questions:
    if should_expand_retrieval(q):
        print(f"'{q}' => Khái quát! Chỉnh k = {base_k + 4}")
    else:
        print(f"'{q}' => Cụ thể! Giữ nguyên k = {base_k}")

print("\n" + "-"*40 + "\n")

# 2. Kỹ thuật XỬ LÝ METADATA (Mô phỏng Citation)
from langchain_core.documents import Document
doc = Document(
    page_content="Text mẫu...", 
    metadata={"source": "File_PDFs/Bản sao của BanMOTa_CNTT_2020-2024.pdf", "page_number": 12}
)

def pretty_source_name(metadata):
    import os
    src = metadata.get("source", "Tài liệu")
    filename = os.path.basename(src)
    page = metadata.get("page_number")
    return f"[{filename} - Trang {page}]" if page else f"[{filename}]"

print("Trích dẫn: ", pretty_source_name(doc.metadata))

'Điều 5 quy chế học vụ nói gì?' => Cụ thể! Giữ nguyên k = 4
'Hãy liệt kê các mục tiêu của môn Cấu trúc dữ liệu' => Khái quát! Chỉnh k = 8

----------------------------------------

Trích dẫn:  [Bản sao của BanMOTa_CNTT_2020-2024.pdf - Trang 12]


## 🎯 Bài tập nhỏ
**Yêu cầu**: 
1. Thêm từ khóa `"tong quan"` (tổng quan) vào biến `broad_markers` của hàm `should_expand_retrieval`.
2. Kiểm tra với câu hỏi: `"Tổng quan về lịch sử trường SGU"`, k có được cộng thêm 4 không?

In [2]:
# 1. Hàm giải quyết bài tập (thêm từ khóa 'tong quan')
def should_expand_retrieval_v2(question: str) -> bool:
    lowered = question.casefold()
    without_marks = "".join(c for c in unicodedata.normalize("NFD", lowered) if unicodedata.category(c) != "Mn")
    normalized = re.sub(r"[^a-z0-9]+", " ", without_marks).strip()
    
    # Dòng quan trọng: Thêm 'tong quan' vào Tuple
    broad_markers = ("muc tieu", "chuan dau ra", "noi dung", "bao gom", "tong quan")
    return any(marker in normalized for marker in broad_markers)

# 2. Kiểm tra với câu hỏi có từ khóa 'tổng quan'
test_q = "Tổng quan về lịch sử trường SGU"
base_k = 4

print(f"Câu hỏi test: '{test_q}'")
if should_expand_retrieval_v2(test_q):
    print(f"=> CÓ chứa từ khóa khái quát! Số tài liệu truy xuất là: k = {base_k + 4}")
else:
    print(f"=> KHÔNG chứa từ khóa khái quát! Số tài liệu truy xuất là: k = {base_k}")


Câu hỏi test: 'Tổng quan về lịch sử trường SGU'
=> CÓ chứa từ khóa khái quát! Số tài liệu truy xuất là: k = 8
